# SmartCampus AI — Multimodal Evaluation: VLM OCR (`1.4 Reader`) & Whisper STT (`1.1 Listener`)

**Project:** SmartCampus AI (IIT / ISB + ISA extension, Groupe 3 *Night Agents*, NAU 2026–2027)
**Recommended Colab Pro Runtime:** GPU `L4 (24 GB)` (Account #5)

This notebook evaluates our local open-source <= 8B multimodal stack:
1. **VLM OCR (`VISION 1.4 Reader`):** Zero-Shot & Few-Shot extraction of bilingual French/Arabic Tunisian Baccalaureate & university Licence transcripts (images and PDFs) comparing **`Qwen/Qwen2.5-VL-7B-Instruct`** (7.6B, Apache 2.0) vs **`google/gemma-3-4b-it`** (4.3B) against the `>= 95%` field-level exact match target.
2. **Whisper STT (`VISION 1.1 Listener`):** Local keyless Word Error Rate (`WER`) baseline on Tunisian Derja / French / code-switched Visitor and Candidate voice notes using **`openai/whisper-large-v3`**.


In [ ]:
# Cell 1 — Mount Google Drive (when on Colab) & install dependencies
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Running outside Google Colab; using local workspace paths.")

!pip install -q bitsandbytes accelerate transformers qwen-vl-utils pillow pypdf pypdfium2 jiwer openai-whisper pandas


In [ ]:
# Cell 2 — Configuration & Paths
import io
import json
import re
import time
import unicodedata
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from pypdf import PdfReader

DRIVE_ROOT = Path("/content/drive/MyDrive/SmartCampus_AI")

def _discover_repo_root() -> Path | None:
    for cand in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path("/content/Projet_IA_Generative")):
        if (cand / "backend" / "tests" / "data" / "sample_bac").exists():
            return cand
    return None

REPO_ROOT = _discover_repo_root()
DRIVE_TRANSCRIPTS_DIR = DRIVE_ROOT / "eval_data" / "transcripts"
REPO_SAMPLE_BAC_DIR = (REPO_ROOT / "backend" / "tests" / "data" / "sample_bac") if REPO_ROOT else None
TRANSCRIPTS_DIR = (
    DRIVE_TRANSCRIPTS_DIR
    if (DRIVE_TRANSCRIPTS_DIR / "ground_truth.json").exists()
    else (REPO_SAMPLE_BAC_DIR if REPO_SAMPLE_BAC_DIR and REPO_SAMPLE_BAC_DIR.exists() else DRIVE_TRANSCRIPTS_DIR)
)
AUDIO_DIR = DRIVE_ROOT / "eval_data" / "voice_notes"  # Put anonymized WAV/MP3 + ground_truth.json here
OUT_DIR = (DRIVE_ROOT / "benchmarks") if Path("/content/drive/MyDrive").exists() else Path("benchmarks")
OUT_DIR.mkdir(parents=True, exist_ok=True)

VLM_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"  # Compare against: "google/gemma-3-4b-it"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("VLM model:", VLM_MODEL_ID)
print("Transcripts dir:", TRANSCRIPTS_DIR)


In [ ]:
# Cell 3 — Load Open-Source VLM (4-bit NF4, AutoModelForImageTextToText) & Aligned JSON OCR Prompt
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import AutoModelForImageTextToText
except ImportError:
    AutoModelForImageTextToText = None
try:
    from transformers import Qwen2_5_VLForConditionalGeneration
except ImportError:
    Qwen2_5_VLForConditionalGeneration = None

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_vlm_model(model_id: str):
    if AutoModelForImageTextToText is not None:
        try:
            return AutoModelForImageTextToText.from_pretrained(
                model_id,
                quantization_config=bnb_cfg,
                device_map="auto",
            )
        except (ValueError, KeyError) as exc:
            if "qwen2.5-vl" in model_id.lower() and Qwen2_5_VLForConditionalGeneration is not None:
                print(f"Falling back to Qwen2_5_VLForConditionalGeneration ({exc})")
            else:
                raise
    if Qwen2_5_VLForConditionalGeneration is not None:
        return Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_cfg,
            device_map="auto",
        )
    raise ImportError("Install a transformers version providing AutoModelForImageTextToText.")

processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)
vlm_model = load_vlm_model(VLM_MODEL_ID)

# Aligned with backend/app/services/vlm_transcript_service.py::VLM_TRANSCRIPT_SYSTEM_PROMPT & schema
OCR_SYSTEM_PROMPT = (
    "You are VISION 1.4_Reader, a high-precision VLM/OCR extractor for official Tunisian "
    "academic transcripts for SmartCampus AI (IIT / ISB / ISA — NAU Sfax).\n"
    "You support TWO official transcript types:\n"
    "A. Tunisian Baccalaureate Transcript (document_type='RELEVE_BAC', diploma_stage='BAC', "
    "Arabic RTL 'بيان أعداد امتحان البكالوريا'):\n"
    "   - Map Section ('الشعبة') strictly: الرياضيات=Mathématiques | العلوم التجريبية=Sciences Expérimentales | "
    "العلوم التقنية=Sciences Techniques | علوم الإعلامية=Sciences de l'Informatique | "
    "الاقتصاد والتصرف=Économie et Gestion | الآداب=Lettres | الرياضة=Sport.\n"
    "   - Translate Arabic subjects accurately (never confuse انقليزية=Anglais with إعلامية=Informatique): "
    "رياضيات=Mathématiques, علوم فيزيائية=Sciences Physiques, علوم الحياة والأرض=Sciences de la Vie et de la Terre, "
    "انقليزية=Anglais, فرنسية=Français, عربية=Arabe, فلسفة=Philosophie, إعلامية=Informatique, "
    "تربية بدنية=Éducation Physique, إسبانية=Espagnol (Option).\n"
    "   - Read governorate from 'مكانها' (e.g. صفاقس -> Sfax), not Ministry issue city 'تونس في'.\n"
    "   - If coefficients are not printed, set coefficient=null so the backend applies official Ministry coefficients.\n"
    "   - Parse decimal commas (e.g. '09,25' -> 9.25); set is_optional=true for optional subjects "
    "(إسبانية, ألمانية, إيطالية, صينية, روسية, موسيقى, تربية تشكيلية).\n"
    "   - Map Arabic mentions: متوسط=Passable, قريب من الحسن=Assez Bien, حسن=Bien, حسن جدا=Très Bien.\n"
    "   - INPDP anonymization rule: if personal data (name, CIN, signature) is masked/blacked out, "
    "return strictly null for student_name and NEVER guess.\n"
    "B. University / Licence Passerelle Transcript (document_type='RELEVE_UNIVERSITAIRE', "
    "diploma_stage='L1'|'L2'|'L3'|'PREPA_1'|'PREPA_2'|'M1'|'OTHER'):\n"
    "   - Extract Teaching Units (subjects with grade, coefficient, credits_acquired), general_average, "
    "credits_acquired, program_title, mention, and jury_decision (preferring Rattrapage if present/higher).\n\n"
    "STRICT RULES:\n"
    "1. Extract ONLY visibly printed values. Never invent unreadable grades, subjects, or averages.\n"
    "2. Return ONLY a valid JSON object matching this exact schema:\n"
    "{\n"
    '  "document_type": "RELEVE_BAC | RELEVE_UNIVERSITAIRE",\n'
    '  "diploma_stage": "BAC | L1 | L2 | L3 | PREPA_1 | PREPA_2 | M1 | OTHER",\n'
    '  "bac_section": "Mathématiques | Sciences Expérimentales | Sciences Techniques | '
    'Sciences de l\'Informatique | Économie et Gestion | Lettres | Sport | Autre | null",\n'
    '  "program_title": "University diploma title or null",\n'
    '  "session_year": "2021 or 2023-2024",\n'
    '  "session_type": "Principale | Contrôle | Rattrapage | Inconnue",\n'
    '  "student_name": null,\n'
    '  "institution": "High School / Faculty or null",\n'
    '  "governorate": "Governorate (e.g. Sfax) or null",\n'
    '  "general_average": 11.51,\n'
    '  "credits_acquired": null,\n'
    '  "mention": "Passable | Assez Bien | Bien | Très Bien | null",\n'
    '  "jury_decision": "Admis | Refusé | Contrôle | null",\n'
    '  "subjects": [\n'
    '    {"name": "Mathématiques", "grade": 9.25, "coefficient": 4.0, '
    '"credits_acquired": null, "is_optional": false, "confidence": 0.96}\n'
    "  ],\n"
    '  "field_confidences": {\n'
    '    "bac_section": 0.98, "session_year": 0.98, "general_average": 0.97, '
    '"student_name": null, "institution": null\n'
    "  }\n"
    "}\n"
    "3. Assign confidence < 0.85 for any blurry, occluded, or uncertain field so a human reviews it (HITL).\n"
    "4. All grades must be numeric on a 0.0 to 20.0 scale."
)

def inspect_transcript_file(file_path: Path) -> dict:
    """Inspect image or PDF (using pypdf text + embedded scan extraction, plus optional pypdfium2 page render)."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        reader = PdfReader(str(file_path))
        pages_text = [(pg.extract_text() or "").strip() for pg in reader.pages]
        pdf_text = "\n".join(t for t in pages_text if t)
        pil_img = None
        for pg in reader.pages:
            if pg.images:
                pil_img = Image.open(io.BytesIO(pg.images[0].data)).convert("RGB")
                break
        if pil_img is None:
            try:
                import pypdfium2 as pdfium
                pdf_doc = pdfium.PdfDocument(str(file_path))
                pil_img = pdf_doc[0].render(scale=2.0).to_pil().convert("RGB")
            except Exception:
                pil_img = None
        return {
            "file": file_path.name,
            "kind": "pdf",
            "pages": len(reader.pages),
            "pdf_text": pdf_text,
            "image": pil_img,
            "image_size": pil_img.size if pil_img is not None else None,
        }
    pil_img = Image.open(file_path).convert("RGB")
    return {
        "file": file_path.name,
        "kind": "image",
        "pages": 1,
        "pdf_text": "",
        "image": pil_img,
        "image_size": pil_img.size,
    }

def extract_transcript_json(file_path: Path) -> tuple[str, float]:
    info = inspect_transcript_file(file_path)
    img = info["image"]
    pdf_text = info["pdf_text"]

    user_content = []
    images_arg = None
    if img is not None:
        img = img.copy()
        img.thumbnail((1280, 1280))
        user_content.append({"type": "image", "image": img})
        images_arg = [img]
    prompt_text = (
        "Extract document_type, diploma_stage, bac_section, program_title, session_year, session_type, "
        "student_name (null if anonymized), institution, governorate, general_average, credits_acquired, "
        "mention, jury_decision, all subjects (with is_optional and confidence), and field_confidences as strict JSON."
    )
    if pdf_text:
        prompt_text += f"\n\n[EXTRACTED PDF TEXT]\n{pdf_text[:6000]}"
    user_content.append({"type": "text", "text": prompt_text})

    messages_structured = [
        {"role": "system", "content": [{"type": "text", "text": OCR_SYSTEM_PROMPT}]},
        {"role": "user", "content": user_content},
    ]
    try:
        text = processor.apply_chat_template(messages_structured, tokenize=False, add_generation_prompt=True)
    except Exception:
        messages_plain = [
            {"role": "system", "content": OCR_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
        text = processor.apply_chat_template(messages_plain, tokenize=False, add_generation_prompt=True)

    proc_kwargs = {"text": [text], "padding": True, "return_tensors": "pt"}
    if images_arg is not None:
        proc_kwargs["images"] = images_arg
    inputs = processor(**proc_kwargs).to(vlm_model.device)

    t0 = time.perf_counter()
    with torch.no_grad():
        # Omit temperature when do_sample=False to avoid transformers generation validation warnings/errors
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=1536, do_sample=False)
    latency_s = round(time.perf_counter() - t0, 3)
    trimmed = generated_ids[:, inputs["input_ids"].shape[1]:]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()
    return output_text, latency_s


In [ ]:
# Cell 4 — PDF/Image Inspection (pypdf) & Subject-Level + Header Field-Level Exact Match Evaluation
ARABIC_TO_CANONICAL_SUBJECT = {
    "رياضيات": "Mathématiques", "الرياضيات": "Mathématiques",
    "علوم فيزيائية": "Sciences Physiques", "العلوم الفيزيائية": "Sciences Physiques",
    "علوم الحياة والأرض": "Sciences de la Vie et de la Terre", "علوم الحياة والارض": "Sciences de la Vie et de la Terre",
    "انقليزية": "Anglais", "إنقليزية": "Anglais", "إنجليزية": "Anglais", "انجليزية": "Anglais",
    "فرنسية": "Français", "عربية": "Arabe", "فلسفة": "Philosophie",
    "إعلامية": "Informatique", "اعلامية": "Informatique",
    "تربية بدنية": "Éducation Physique", "تكنولوجيا": "Technologie",
    "إسبانية": "Espagnol (Option)", "اسبانية": "Espagnol (Option)",
}

def _fold(text: str | None) -> str:
    if not text:
        return ""
    canon = ARABIC_TO_CANONICAL_SUBJECT.get(str(text).strip(), str(text).strip())
    decomposed = unicodedata.normalize("NFKD", canon)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch)).lower().strip()

def _parse_float(val) -> float | None:
    if val is None:
        return None
    try:
        return float(str(val).strip().replace(",", "."))
    except ValueError:
        return None

def _parse_vlm_json(raw_out: str) -> dict:
    cleaned = raw_out.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE)
        cleaned = re.sub(r"\s*```$", "", cleaned)
    return json.loads(cleaned.strip())

def evaluate_subject_exact_match(pred_subjects: list[dict], gt_subjects: list[dict], tol: float = 0.05) -> tuple[int, int, float]:
    if not gt_subjects:
        return 0, 0, 1.0
    pred_map: dict[str, dict] = {}
    for s in pred_subjects or []:
        if isinstance(s, dict) and s.get("name"):
            pred_map[_fold(s["name"])] = s
    matched = 0
    for gt in gt_subjects:
        key = _fold(gt["name"])
        pred = pred_map.get(key)
        if pred is None:
            continue
        pred_grade = _parse_float(pred.get("grade"))
        gt_grade = _parse_float(gt.get("grade"))
        grade_ok = pred_grade is not None and gt_grade is not None and abs(pred_grade - gt_grade) <= tol
        opt_ok = ("is_optional" not in gt) or (bool(pred.get("is_optional", False)) == bool(gt["is_optional"]))
        if grade_ok and opt_ok:
            matched += 1
    return matched, len(gt_subjects), round(matched / len(gt_subjects), 4)

# Verified ground truth for the 3 anonymized fixtures in backend/tests/data/sample_bac/
SAMPLE_BAC_DEFAULT_GT = [
    {
        "file": "releve_bac_math_sfax_anonymized.jpg",
        "document_type": "RELEVE_BAC",
        "bac_section": "Mathématiques",
        "general_average": 11.51,
        "student_name": None,
        "subjects": [
            {"name": "Mathématiques", "grade": 9.25, "is_optional": False},
            {"name": "Sciences Physiques", "grade": 11.50, "is_optional": False},
            {"name": "Sciences de la Vie et de la Terre", "grade": 12.50, "is_optional": False},
            {"name": "Anglais", "grade": 13.50, "is_optional": False},
            {"name": "Français", "grade": 8.50, "is_optional": False},
            {"name": "Arabe", "grade": 10.75, "is_optional": False},
            {"name": "Philosophie", "grade": 7.50, "is_optional": False},
            {"name": "Informatique", "grade": 17.00, "is_optional": False},
            {"name": "Éducation Physique", "grade": 18.66, "is_optional": False},
            {"name": "Espagnol (Option)", "grade": 11.25, "is_optional": True},
        ],
    },
    {
        "file": "releve_bac_math_sfax_anonymized.pdf",
        "document_type": "RELEVE_BAC",
        "bac_section": "Mathématiques",
        "general_average": 11.51,
        "student_name": None,
        "subjects": [
            {"name": "Mathématiques", "grade": 9.25, "is_optional": False},
            {"name": "Sciences Physiques", "grade": 11.50, "is_optional": False},
            {"name": "Sciences de la Vie et de la Terre", "grade": 12.50, "is_optional": False},
            {"name": "Anglais", "grade": 13.50, "is_optional": False},
            {"name": "Français", "grade": 8.50, "is_optional": False},
            {"name": "Arabe", "grade": 10.75, "is_optional": False},
            {"name": "Philosophie", "grade": 7.50, "is_optional": False},
            {"name": "Informatique", "grade": 17.00, "is_optional": False},
            {"name": "Éducation Physique", "grade": 18.66, "is_optional": False},
            {"name": "Espagnol (Option)", "grade": 11.25, "is_optional": True},
        ],
    },
    {
        "file": "releve_licence_l3_isims_sfax_anonymized.pdf",
        "document_type": "RELEVE_UNIVERSITAIRE",
        "bac_section": None,
        "general_average": 12.98,
        "student_name": None,
        "subjects": [
            {"name": "SI Décisionnels", "grade": 9.52},
            {"name": "Technologies avancées Big Data", "grade": 10.19},
            {"name": "Unités transversales", "grade": 10.00},
            {"name": "Techniques avancées Big Data", "grade": 11.85},
            {"name": "UOPS1-ADBD", "grade": 8.88},
            {"name": "Stage de fin d'études", "grade": 15.87},
        ],
    },
]

if TRANSCRIPTS_DIR.exists():
    # 1. Inspect all PDFs and images using pypdf + Pillow
    inspected_rows = []
    for fpath in sorted(TRANSCRIPTS_DIR.iterdir()):
        if fpath.suffix.lower() in {".pdf", ".jpg", ".jpeg", ".png", ".webp"}:
            meta = inspect_transcript_file(fpath)
            inspected_rows.append({
                "file": meta["file"],
                "kind": meta["kind"],
                "pages": meta["pages"],
                "pdf_text_chars": len(meta["pdf_text"]),
                "image_size": str(meta["image_size"]),
            })
    print("Inspected transcript files (pypdf + Pillow):")
    print(pd.DataFrame(inspected_rows).to_string(index=False))

    gt_file = TRANSCRIPTS_DIR / "ground_truth.json"
    if gt_file.exists():
        gt_items = json.loads(gt_file.read_text(encoding="utf-8"))
    else:
        gt_items = [g for g in SAMPLE_BAC_DEFAULT_GT if (TRANSCRIPTS_DIR / g["file"]).exists()]

    if gt_items:
        results = []
        for item in gt_items:
            doc_path = TRANSCRIPTS_DIR / item["file"]
            raw_out, lat = extract_transcript_json(doc_path)
            try:
                parsed = _parse_vlm_json(raw_out)
                pred_avg = _parse_float(parsed.get("general_average"))
                gt_avg = _parse_float(item.get("general_average"))
                avg_match = pred_avg is not None and gt_avg is not None and abs(pred_avg - gt_avg) <= 0.05
                gt_sec = item.get("bac_section")
                sec_match = (parsed.get("bac_section") is None) if gt_sec is None else (_fold(parsed.get("bac_section")) == _fold(gt_sec))
                doc_type_match = str(parsed.get("document_type", "")).upper() == str(item.get("document_type", "RELEVE_BAC")).upper()
                inpdp_null_ok = (parsed.get("student_name") is None) if item.get("student_name") is None else True
                subj_matched, subj_total, subj_rate = evaluate_subject_exact_match(
                    parsed.get("subjects", []), item.get("subjects", [])
                )
                header_checks = [avg_match, sec_match, doc_type_match, inpdp_null_ok]
                total_fields = len(header_checks) + subj_total
                overall_exact = round((sum(header_checks) + subj_matched) / max(1, total_fields), 4)
            except Exception as exc:
                avg_match = sec_match = doc_type_match = inpdp_null_ok = False
                subj_matched, subj_total, subj_rate, overall_exact = 0, len(item.get("subjects", [])), 0.0, 0.0
                print(f"JSON parse error on {item['file']}: {exc}")

            results.append({
                "model": VLM_MODEL_ID,
                "file": item["file"],
                "latency_s": lat,
                "document_type_exact": doc_type_match,
                "general_average_exact": avg_match,
                "bac_section_exact": sec_match,
                "inpdp_student_name_null_ok": inpdp_null_ok,
                "subjects_matched": f"{subj_matched}/{subj_total}",
                "subject_exact_match_rate": subj_rate,
                "overall_field_exact_match": overall_exact,
            })
        df_vlm = pd.DataFrame(results)
        out_csv = OUT_DIR / f"vlm_ocr_{VLM_MODEL_ID.replace('/', '__')}.csv"
        df_vlm.to_csv(out_csv, index=False)
        print(f"Mean overall field exact match: {df_vlm['overall_field_exact_match'].mean():.4f} | Saved: {out_csv}")
        df_vlm
    else:
        print("Place anonymized transcripts + ground_truth.json in:", TRANSCRIPTS_DIR)
else:
    print("Place anonymized transcripts + ground_truth.json in:", TRANSCRIPTS_DIR)


In [ ]:
# Cell 5 — Whisper STT (local openai/whisper-large-v3) WER Benchmark on Visitor / Candidate Voice Notes
import whisper
from jiwer import wer

whisper_gt = AUDIO_DIR / "ground_truth.json"
if whisper_gt.exists():
    stt_model = whisper.load_model("large-v3")
    items = json.loads(whisper_gt.read_text(encoding="utf-8"))
    stt_rows = []
    for item in items:
        audio_path = AUDIO_DIR / item["file"]
        t0 = time.perf_counter()
        res = stt_model.transcribe(
            str(audio_path),
            initial_prompt="SmartCampus NAU Sfax IIT ISB ISA Licence Prépa Ingénieur GLID SDIA عسلامة شنوة كيفاش",
        )
        lat = round(time.perf_counter() - t0, 3)
        hyp = res["text"].strip()
        ref = item["reference_text"].strip()
        stt_rows.append({
            "file": item["file"],
            "latency_s": lat,
            "wer": round(wer(ref, hyp), 4),
            "reference": ref,
            "hypothesis": hyp,
        })
    df_stt = pd.DataFrame(stt_rows)
    out_stt = OUT_DIR / "whisper_large_v3_wer.csv"
    df_stt.to_csv(out_stt, index=False)
    print(f"Mean WER: {df_stt['wer'].mean():.4f} | Saved:", out_stt)
    df_stt[["file", "latency_s", "wer"]]
else:
    print("Place anonymized audio notes + ground_truth.json in:", AUDIO_DIR)
